In [ ]:
import os
import pandas as pd
import numpy as np
import copy
import torch
import math
from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import confusion_matrix, accuracy_score, cohen_kappa_score, f1_score, classification_report
import matplotlib.pyplot as plt
import seaborn as sns

# --- CONFIGURATION ---
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
PATH = '/kaggle/input/datasets/anisnourreddine/projet-rn' 
FILES = ['MCTNet_Arkansas_2021(reboot).csv'] 

N_DATES = 36 
BATCH_SIZE = 32 

print(f"Appareil : {DEVICE} | Étude : Arkansas 2021")

In [ ]:
class CropDataset(Dataset):
    def __init__(self, x, y, m):
        self.x = torch.FloatTensor(x) #spectral features
        self.y = torch.LongTensor(y) #class labels 
        self.m = torch.FloatTensor(m) #validity masks
    def __len__(self): return len(self.y)
    def __getitem__(self, idx): return self.x[idx], self.y[idx], self.m[idx] #to build batches

def load_and_split_Arkansas(files):
    TARGET_BANDS = ['Blue', 'Green', 'Red', 'RE1', 'RE2', 'RE3', 'NIR', 'RE4', 'SWIR1', 'SWIR2']

    MY_LABELS = {
        0: 'Soybeans', 1: 'Rice', 2: 'Corn', 3: 'Cotton', 4: 'Others'
    }

    X_list, y_list, M_list = [], [], []

    for f in files:
        full_path = os.path.join(PATH, f)
        if not os.path.exists(full_path):
            print(f"Fichier introuvable : {full_path}")
            continue

        df = pd.read_csv(full_path)

        n_before   = len(df)
        valid_mask = df.notna().all(axis=1)
        df         = df[valid_mask].reset_index(drop=True)
        n_dropped  = n_before - len(df)
        print(f"  [{f}] Dropped {n_dropped}/{n_before} rows with NaN "
              f"({n_dropped/n_before:.1%}) — {len(df)} rows remaining.")

        temp_X = []
        for b in TARGET_BANDS:
            cols = [f"{b}_t{t:02d}" for t in range(1, 37)] #on prend les colonnes de chaque bande pour les 36 dates Blue_t01 through Blue_t36, then Green_t01 through Green_t36, and so on. 
            temp_X.append(df[cols].values) # on cree 10 matrcies de tailles (N , 36) lignes : pixels , colonnes : 36 dates pour chaque bande
        X_3d = np.stack(temp_X, axis=2)  # on stack les 10 matrices pour obtenir une matrice 3D de taille (N, 36, 10) : N pixels, 36 dates, 10 bandes


        y_names = df['label'].apply(lambda x: MY_LABELS.get(x, 'Others')).values

        M_2d = df[[f"valid_t{t:02d}" for t in range(1, 37)]].values #This extracts the binary mask: 1 means the satellite had clear sky on that date and the spectral values are reliable. 0 means clouds or cloud shadow corrupted the observation

        X_list.append(X_3d)
        y_list.append(y_names)
        M_list.append(M_2d)

    X_all = np.vstack(X_list)
    y_all_raw = np.concatenate(y_list)
    M_all = np.vstack(M_list)

    le = LabelEncoder() #cnn doesnt work with strings so we convert the label names into numerical values
    le.fit(list(MY_LABELS.values()))
    y_all = le.transform(y_all_raw)

    train_idx, val_idx, test_idx = [], [], []

    np.random.seed(69)

    for c in np.unique(y_all): #our chacune des classes labelisées
        indices = np.where(y_all == c)[0]
        np.random.shuffle(indices) #melanger les indices pour chaque classe pour éviter tout biais de tri

        n_train = min(len(indices), 240) #on prend 240 echatnillons par classe pour train
        n_val   = min(len(indices) - n_train, 60) # 60 echantillons pour val

        train_idx.extend(indices[:n_train])
        val_idx.extend(indices[n_train:n_train + n_val])
        test_idx.extend(indices[n_train + n_val:])

    return (X_all[train_idx], y_all[train_idx], M_all[train_idx],
            X_all[val_idx],   y_all[val_idx],   M_all[val_idx],
            X_all[test_idx],  y_all[test_idx],  M_all[test_idx], le)

# --- EXÉCUTION ---
X_train, y_train, M_train, X_val, y_val, M_val, X_test, y_test, M_test, encoder = load_and_split_Arkansas(FILES)

train_loader = DataLoader(CropDataset(X_train, y_train, M_train), batch_size=BATCH_SIZE, shuffle=True) #shuffle=True pour mélanger les données à chaque époque.
val_loader   = DataLoader(CropDataset(X_val,   y_val,   M_val),   batch_size=BATCH_SIZE)
test_loader  = DataLoader(CropDataset(X_test,  y_test,  M_test),  batch_size=BATCH_SIZE)

print(f"Echantillons : Train={len(X_train)}, Val={len(X_val)}, Test={len(X_test)}")
print(f"Ordre des classes : {encoder.classes_}")


In [ ]:
class ECA(nn.Module):
    # ECA answers one question: which spectral bands matter most for this input?
    def __init__(self, k_size=3):
        super().__init__()
        #notre input est la matrice 3D de taille (N, 36, 10) : N pixels, 36 dates, 10 bandes
        self.avg_pool = nn.AdaptiveAvgPool1d(1) #on reduit la dimension en (B =N, canals = 10, moyennes des time stamps =1) on prond les moyennes des valeurs des bandes au courant des 36 dates
        self.conv = nn.Conv1d(1, 1, kernel_size=k_size, padding=(k_size - 1) // 2, bias=False) #compare chaque bande a ses deux voisins simulatneament pour apprendre les interactions entre bandes voisines (ex: Red et NIR sont souvent corrélées pour la végétation)
        self.sigmoid = nn.Sigmoid() #quashes the output to [0, 1]

    def forward(self, x):
        # x: [Batch, Canaux, Temps]
        y = self.avg_pool(x)
        y = self.conv(y.transpose(-1, -2)).transpose(-1, -2)   # On transpose pour que la convolution agisse sur les canaux et pas le temps
        return x * self.sigmoid(y)  
        #Multiplying by the original x applies the learned gates: 
        # channels with a gate close to 1 pass through unchanged; 
        # channels with a gate close to 0 are suppressed. 
        # This is soft selection, not hard filtering — every band still contributes, but the model controls how much.

In [ ]:
class ALPE(nn.Module):
    def __init__(self, d_model, max_len=36):
        super().__init__()
        
        pe = torch.zeros(max_len, d_model) # Création de la matrice de sinus/cosinus (Position Encoding classique)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1) # On crée un vecteur colonne allant de 0 à 35 (l'index du temps)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0)) 
        #cree des valeurs unniques de sinus cosinus pour chaque time stamp des 36 jours de chaque canal
        #donnes une matrice (1, 36, 10)

        self.eca = ECA() # Module d'attention spectrale pour pondérer les positions 
        self.conv_refine = nn.Conv1d(d_model, d_model, kernel_size=3, padding=1, groups=d_model)# Convolution de groupe (Depthwise) pour lisser les relations temporelles localement

    def forward(self, x, mask):
        b, t, c = x.size() # b: batch size, t: temps (36), c: canaux (10)
        pe = self.pe.expand(b, -1, -1)# Duplication du PE pour chaque échantillon du batch
        # On multiplie l'encodage de position par le masque de validité (1=clair, 0=nuage)
        # Si c'est nuageux, l'info de position est effacée

        pe = pe * mask.unsqueeze(-1) 
        # The mask (B, 36) is expanded to (B, 36, 1) and multiplied elementwise into the PE (B, 36, 10). 
        # Wherever mask==0 (cloudy date), the PE for that timestep becomes all zeros.
        
       
        pe_f = self.conv_refine(pe.transpose(1, 2)) 
        #After masking, there are abrupt jumps in the PE
        #a sequence of values, then suddenly zeros for a few cloudy dates, then values again.
        # A depthwise Conv1D (groups=d_model means each channel is convolved independently) smooths these transitions. 
        # It "fills in" the gaps with interpolated positional information from neighbouring valid dates.
        pe_f = self.eca(pe_f).transpose(1, 2) # L'ECA sélectionne les bandes de l'encodage les plus utiles pour la classification.
        return pe_f # Retourne le PE adapté qui sera ajouté à l'entrée du Transformer

In [ ]:
class CNNSubModule(nn.Module):
    def __init__(self, in_dim, out_dim, kernel_size=3):
        super().__init__()
        pad = kernel_size // 2
        self.conv1    = nn.Conv1d(in_dim,  out_dim, kernel_size, padding=pad)
        self.bn1      = nn.BatchNorm1d(out_dim)
        self.conv2    = nn.Conv1d(out_dim, out_dim, kernel_size, padding=pad)
        self.bn2      = nn.BatchNorm1d(out_dim)
        self.shortcut = nn.Conv1d(in_dim, out_dim, kernel_size=1)
        self.relu     = nn.ReLU()

    def forward(self, x):
        h = x.transpose(1, 2) # (B, T, C) → (B, C, T) for Conv1d
        residual = self.shortcut(h) # project input to out_dim
        h = self.bn1(self.conv1(h))
        h = self.bn2(self.conv2(h))
        return self.relu(h + residual)


class TransformerSubModule(nn.Module):
    def __init__(self, d_model, n_head, dropout=0.1):
        super().__init__()
        self.mha   = nn.MultiheadAttention(d_model, n_head, dropout=dropout, batch_first=True)
        self.norm1 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_model * 4),
            nn.ReLU(),
            nn.Linear(d_model * 4, d_model)
        )
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x, pe=None):
        x_pe = x + pe if pe is not None else x
        attn_out, _ = self.mha(x_pe, x_pe, x_pe)
        x2 = self.norm1(attn_out + x_pe)   # Add & Norm
        ffn_out = self.ffn(x2)
        x3 = self.norm2(ffn_out + x2)       # Add & Norm
        return x3


class CTFusionBlock(nn.Module):
    def __init__(self, in_dim, out_dim, n_head=5, dropout=0.1):
        super().__init__()
        self.transformer = TransformerSubModule(in_dim, n_head, dropout)
        self.cnn         = CNNSubModule(in_dim, out_dim)
        self.fusion_conv = nn.Conv1d(out_dim + in_dim, out_dim, kernel_size=1)
        self.pool        = nn.MaxPool1d(2)

    def forward(self, x, pe=None):
        t_out = self.transformer(x, pe)

        c_out = self.cnn(x)

        fused = torch.cat([c_out, t_out.transpose(1, 2)], dim=1)

        fused = self.fusion_conv(fused)

        return self.pool(fused).transpose(1, 2)


In [ ]:
class MCTNet(nn.Module):
    """
    MCTNet : 3 stages CTFusion + Global Max Pooling + MLP classifier.
    FIX 7: AdaptiveAvgPool1d remplacé par AdaptiveMaxPool1d (papier section 2.3.2).
    """
    def __init__(self, n_classes):
        super().__init__()
        self.alpe   = ALPE(d_model=10, max_len=36)

        self.stage1 = CTFusionBlock(10,  20,  n_head=5)
        self.stage2 = CTFusionBlock(20,  40,  n_head=5)
        self.stage3 = CTFusionBlock(40,  80,  n_head=5)

        self.global_pool = nn.AdaptiveMaxPool1d(1)
        self.classifier  = nn.Linear(80, n_classes)

    def forward(self, x, mask):
        pe = self.alpe(x, mask)
        x  = self.stage1(x, pe=pe)
        x  = self.stage2(x)
        x  = self.stage3(x)
        x  = self.global_pool(x.transpose(1, 2)).squeeze(-1)
        return self.classifier(x)


In [ ]:
n_classes = len(encoder.classes_)
model     = MCTNet(n_classes).to(DEVICE)

EPOCHS = 200
patience_counter = 0
PATIENCE = 20

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)


criterion = nn.CrossEntropyLoss()

history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
best_val_acc   = 0.0
best_model_wts = copy.deepcopy(model.state_dict())

for epoch in range(EPOCHS):
    model.train()
    running_loss, train_correct = 0.0, 0
    for b_x, b_y, b_m in train_loader:
        b_x, b_y, b_m = b_x.to(DEVICE), b_y.to(DEVICE), b_m.to(DEVICE)
        optimizer.zero_grad()
        out  = model(b_x, b_m)
        loss = criterion(out, b_y)
        loss.backward()
        optimizer.step()
        running_loss  += loss.item()
        train_correct += (out.argmax(1) == b_y).sum().item()

    model.eval()
    v_loss, v_corr = 0.0, 0
    with torch.no_grad():
        for vx, vy, vm in val_loader:
            vx, vy, vm = vx.to(DEVICE), vy.to(DEVICE), vm.to(DEVICE)
            out     = model(vx, vm)
            v_loss += criterion(out, vy).item()
            v_corr += (out.argmax(1) == vy).sum().item()

    epoch_train_acc = train_correct / len(X_train)
    epoch_val_acc   = v_corr       / len(X_val)

    history['train_loss'].append(running_loss / len(train_loader))
    history['val_loss'].append(v_loss         / len(val_loader))
    history['train_acc'].append(epoch_train_acc)
    history['val_acc'].append(epoch_val_acc)


    if epoch_val_acc > best_val_acc:
        best_val_acc = epoch_val_acc
        best_model_wts = copy.deepcopy(model.state_dict())
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"Early stopping at epoch {epoch+1}")
            break

    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/{EPOCHS} | Val Acc: {epoch_val_acc:.2%} | Val Loss: {v_loss / len(val_loader):.4f}")

model.load_state_dict(best_model_wts)
print(f"Meilleure précision validation : {best_val_acc:.4f}")

In [ ]:
def plot_history(history):
    plt.figure(figsize=(15, 5))

    # Graphique de la Perte (Loss)
    plt.subplot(1, 2, 1)
    plt.plot(history['train_loss'], label='Train Loss', color='#1f77b4', lw=2)
    plt.plot(history['val_loss'], label='Val Loss', color='#ff7f0e', lw=2)
    plt.title('Évolution de la Perte (Loss)', fontsize=14, fontweight='bold')
    plt.xlabel('Époques')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.7)

    # Graphique de la Précision (Accuracy)
    plt.subplot(1, 2, 2)
    plt.plot(history['train_acc'], label='Train Acc', color='#1f77b4', lw=2)
    plt.plot(history['val_acc'], label='Val Acc', color='#ff7f0e', lw=2)
    plt.title('Évolution de la Précision (Accuracy)', fontsize=14, fontweight='bold')
    plt.xlabel('Époques')
    plt.ylabel('Précision (%)')
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.7)

    plt.tight_layout()
    plt.show()

# Lancement de la visualisation
plot_history(history)

In [ ]:
def evaluate_mctnet_california_final(model, test_loader, device, encoder):
    model.eval()
    all_y, all_p = [], []
    with torch.no_grad():
        for b_x, b_y, b_m in test_loader:
            outputs = model(b_x.to(device), b_m.to(device))
            all_y.extend(b_y.numpy())
            all_p.extend(outputs.argmax(1).cpu().numpy())
    all_y, all_p = np.array(all_y), np.array(all_p)

    kappa = cohen_kappa_score(all_y, all_p)
    oa    = accuracy_score(all_y, all_p)
    f1    = f1_score(all_y, all_p, average='macro')
    print(f"\nOverall Accuracy (OA) : {oa:.4f}")
    print(f"Cohen's Kappa         : {kappa:.4f}")
    print(f"F1-Score (Macro)      : {f1:.4f}")

    # [FIX] Ordre des classes imposé indépendamment de l'encodeur
    # encoder.classes_ est toujours alphabétique quel que soit le fix appliqué au chargement
    # On reconstruit le bon ordre : le label numérique vient de MY_LABELS {0:Grapes, 1:Rice...}
    # Avant : class_names = encoder.classes_ (alphabétique)
    # Avant : label_order = list(range(len(encoder.classes_)))
    DISPLAY_ORDER = ['Corn', 'Cotton', 'Soybeans', 'Rice', 'Others']
    label_order   = [list(encoder.classes_).index(name) for name in DISPLAY_ORDER]

    cm = confusion_matrix(all_y, all_p, labels=label_order, normalize='true')

    plt.figure(figsize=(12, 10))
    sns.heatmap(cm, annot=True, fmt='.2f', cmap='Greens',
                xticklabels=DISPLAY_ORDER,
                yticklabels=DISPLAY_ORDER)
    plt.title(f'Matrice de Confusion Californie 2021\nAccuracy: {oa:.2%} | Kappa: {kappa:.3f}',
              fontsize=14, fontweight='bold')
    plt.ylabel('Vérité Terrain (USDA)', fontsize=12)
    plt.xlabel('Prédiction MCTNet', fontsize=12)
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()

    print("\n" + "="*60)
    print("Détails par classe :")
    print(classification_report(all_y, all_p,
                                labels=label_order,
                                target_names=DISPLAY_ORDER))
    print("="*60)

evaluate_mctnet_california_final(model, test_loader, DEVICE, encoder)